## Import libraries

In [1]:
import pandas as pd
import json
import os
import re
from collections import defaultdict

In [2]:
INPUT_DIR = './data'
df = pd.read_json(f'{INPUT_DIR}/compile_results_phase_2.jsonl', lines=True)
metadata_df = pd.read_json(f'{INPUT_DIR}/metadata.jsonl', lines=True)

df['temp_id'] = df['sub_id'].str.rsplit('_', n=1).str[0]

cols_to_get = ['id', 'description', 'time_limit', 'memory_limit']

df = df.merge(
    metadata_df[cols_to_get],
    left_on='temp_id',
    right_on='id',
    how='left'
)
df = df.drop(columns=['temp_id'])
df.head()

,sub_id,code,lang,correct,runnable,exec_time_ms,std_error,convention,testcases,memory_mb,id,description,time_limit,memory_limit
0,847_J_c0,#include <bits/stdc++.h>\nusing namespace std;...,cpp,1,True,6.884336,None,Done processing prog.cpp\nTotal errors found: ...,"[{'input': '1 0 ', 'output': '0 '}]",0.125,847_J,Soon the first year students will be initiated...,2000,256
1,847_J_c1,#include <bits/stdc++.h>\nusing namespace std;...,cpp,1,True,4.058599,None,Done processing prog.cpp\nTotal errors found: ...,"[{'input': '1 0 ', 'output': '0 '}]",0.000,847_J,Soon the first year students will be initiated...,2000,256
2,847_J_c2,#include <bits/stdc++.h>\nusing namespace std;...,cpp,1,True,18.798590,None,Done processing prog.cpp\nTotal errors found: ...,"[{'input': '1 0 ', 'output': '0 '}]",0.000,847_J,Soon the first year students will be initiated...,2000,256
3,847_J_c3,#include <bits/stdc++.h>\nusing namespace std;...,cpp,1,True,3.453732,None,Done processing prog.cpp\nTotal errors found: ...,"[{'input': '1 0 ', 'output': '0 '}]",0.000,847_J,Soon the first year students will be initiated...,2000,256
4,847_J_c4,#include <bits/stdc++.h>\nusing namespace std;...,cpp,1,True,5.728722,None,Done processing prog.cpp\nTotal errors found: ...,"[{'input': '1 0 ', 'output': '0 '}]",0.000,847_J,Soon the first year students will be initiated...,2000,256


## Preprocess exec-time

In [3]:
df['optimized_exec_time'] = df.groupby('id')['exec_time_ms'].transform('min')
df['exec_time_ratio'] = df['exec_time_ms']/df['optimized_exec_time']

## Preprocess Convention

In [4]:
LINTER_MAP = {
    'cpp': 'cpplint',
    'py3': 'ruff',
    'java': "Google's java checkstyle"
}
def parse_cpplint(raw_text):
    errors = defaultdict(list)
    pattern = r'prog\.\w+:(\d+):\s+(.+?)\s+\[([^\]]+)\]\s+\[(\d+)\]'
    
    if raw_text is None:
        return dict()
    for match in re.finditer(pattern, raw_text):
        line, message, category, severity = match.groups()
        errors[category].append({
            "line": int(line),
            "message": message.strip(),
        })
    return dict(errors)

def parse_checkstyle(raw_text):
    errors = defaultdict(list)
    pattern = r'\[(\w+)\]\s+[^:]+:(\d+):?(\d+)?:\s+(.+?)\s+\[([^\]]+)\]'
    
    if raw_text is None:
        return dict()
    for match in re.finditer(pattern, raw_text):
        level, line, col, message, checkname = match.groups()
        errors[checkname].append({
            "line": int(line),
            "message": message.strip(),
            "level": level
        })
    return dict(errors)

def parse_ruff(raw_text):
    errors = defaultdict(list)
    ansi_pattern = r'\x1b\[[0-9;]*m'
    pattern = r'([A-Z]+\d+)\s+(.+?)\n\s+-->\s+prog\.py:(\d+):(\d+)'
    text = re.sub(ansi_pattern, '', raw_text)
    
    if raw_text is None:
        return dict()
    for match in re.finditer(pattern, text):
        code, message, line, col = match.groups()
        errors[code].append({
            "line": int(line),
            "message": message.strip(),
        })
    return dict(errors)

def format_for_prompt(parsed_errors, lang):
    if len(parsed_errors.items()) == 0:
        return f"**Convention analysis using {LINTER_MAP[lang]}**: no errors found"

    lines = [f"**Convention analysis using {LINTER_MAP[lang]}**: {sum(len(v) for v in parsed_errors.values())} issues"]
    for cat, errors in parsed_errors.items():
        if lang == 'py3':
            line_refs = '\n'.join(f"\t- L{e['line']}: {e['message']}" for e in errors)
            line_refs = '\n' + line_refs
        else:
            line_numbers = [e['line'] for e in errors]
            line_refs = ", ".join(f"L{l}" for l in line_numbers)
        lines.append(f"- {cat} ({len(errors)} errors) in lines: {line_refs}")
    return "\n".join(lines)

def preprocess_convention(row):
    raw = row['convention']
    lang = row['lang']

    if lang == 'py3':
        errors = parse_ruff(raw)
    elif lang == 'java':
        errors = parse_checkstyle(raw)
    elif lang == 'cpp':
        errors = parse_cpplint(raw)
    else:
        return f"**Convention analysis**: unsupported convention checking tool for {lang}"

    return format_for_prompt(errors, lang)

## Preprocess std_error

In [5]:
ERROR_KEYWORDS = [
    # C/C++ COMPILER ERRORS
    "error:",
    "fatal error",
    "undefined reference",
    "collect2: error",
    "ld returned",
    "no matching function",
    "was not declared",
    "not declared in this scope",
    "no match for 'operator",
    "invalid operands",
    "expected",
    "redefinition of",
    "conflicting types",
    "implicit declaration",
    "incompatible types",
    "cannot convert",
    "invalid conversion",
    "no member named",
    "use of undeclared identifier",
    "too few arguments",
    "too many arguments",
    "ambiguous",
    "extended character",
    
    # C/C++ RUNTIME ERRORS
    "segmentation fault",
    "sigsegv",
    "sigabrt",
    "sigfpe",
    "sigkill",
    "core dumped",
    "stack smashing",
    "buffer overflow",
    "double free",
    "free(): invalid",
    "malloc",
    "munmap_chunk",
    "abort",
    "floating point exception",
    
    # C++ STL Exceptions
    "terminate called",
    "std::bad_alloc",
    "std::out_of_range",
    "std::length_error",
    "std::overflow_error",
    "std::underflow_error",
    "std::runtime_error",
    "std::logic_error",
    "std::invalid_argument",
    "std::domain_error",
    "std::range_error",
    "cannot create std::vector",
    
    # JAVA COMPILE ERRORS
    "cannot find symbol",
    "incompatible types",
    "unreported exception",
    "non-static method",
    "non-static variable",
    "reached end of file while parsing",
    "class, interface, or enum expected",
    "';' expected",
    "illegal start of expression",
    
    # JAVA RUNTIME EXCEPTIONS
    "exception in thread",
    "java.lang.nullpointerexception",
    "java.lang.arrayindexoutofboundsexception",
    "java.lang.stringindexoutofboundsexception",
    "java.lang.numberformatexception",
    "java.lang.arithmeticexception",
    "java.lang.classcastexception",
    "java.lang.illegalargumentexception",
    "java.lang.illegalstateexception",
    "java.lang.unsupportedoperationexception",
    "java.lang.stackoverflowerror",
    "java.lang.outofmemoryerror",
    "java.util.nosuchelementexception",
    "java.util.concurrentmodificationexception",
    "java.io.filenotfoundexception",
    "java.io.ioexception",
    
    # PYTHON ERRORS
    "traceback",
    "syntaxerror",
    "indentationerror",
    "taberror",
    "nameerror",
    "typeerror",
    "valueerror",
    "indexerror",
    "keyerror",
    "attributeerror",
    "zerodivisionerror",
    "overflowerror",
    "recursionerror",
    "memoryerror",
    "runtimeerror",
    "importerror",
    "modulenotfounderror",
    "filenotfounderror",
    "unboundlocalerror",
    "stopiteration",
    "assertionerror",
    "unicodedecodeerror",
    "unicodeencodeerror",
    "eofError",
    
    # SYSTEM/GENERAL ERRORS
    "system_timeout",
    "time limit exceeded",
    "memory limit exceeded",
    "runtime error",
    "compilation error",
    "killed",
    "signal",
]

In [6]:
def preprocess_std_error(std_error, max_length = 500):
    if not std_error or std_error.isspace():
        return ""
    
    std_error = std_error.strip()
    
    if std_error == "system_timeout":
        return ""
    
    std_error = std_error.replace("\\n", "\n").replace("\\t", "\t")
    
    extracted_info = extract_key_error_info(std_error)
    
    if len(extracted_info) > max_length:
        extracted_info = extracted_info[:max_length] + "\n... [truncated]"
    
    return f"**Syntax/Runtime Error**:\n```\n{extracted_info}\n```" if extracted_info else ""


def extract_key_error_info(error_text):
    lines = error_text.split("\n")
    key_lines = []
    seen_errors = set()
    
    for line in lines:
        # Remove note
        if "note:" in line.lower():
            continue
            
        # Keep some main error
        if any(keyword in line.lower() for keyword in ERROR_KEYWORDS):
            simplified = simplify_path(line)
            
            error_key = extract_error_key(simplified)
            if error_key and error_key not in seen_errors:
                seen_errors.add(error_key)
                key_lines.append(simplified)
                
                if len(key_lines) >= 5:
                    break
    
    return "\n".join(key_lines) if key_lines else error_text[:300]


def simplify_path(line):
    # Pattern: /long/path/to/file.cpp:123:45: -> file.cpp:123:
    pattern = r'/[\w/\-\.]+/(\w+\.\w+):(\d+):\d*:?'
    return re.sub(pattern, r'\1:\2: ', line)


def extract_error_key(line):
    # Find main error
    match = re.search(r"error:\s*(.+?)(?:\n|$)", line, re.IGNORECASE)
    if match:
        return match.group(1)[:50]
    return line[:50] if line else ""



In [7]:
df['convention_prompt'] = df.apply(preprocess_convention, axis=1)
df['syntax_error'] = df['std_error'].apply(preprocess_std_error)
df = df.drop(columns=cols_to_get)
df.to_json(f'{INPUT_DIR}/final_data.jsonl', orient='records', lines=True)

In [8]:
print(df['syntax_error'].value_counts())

syntax_error
                                                                                                                                                                                                                                                                    18191
**Syntax/Runtime Error**:\n```\njava.io.FileNotFoundException: input.txt (No such file or directory)\n```                                                                                                                                                              24
**Syntax/Runtime Error**:\n```\nException in thread "main" java.io.FileNotFoundException: input.txt (No such file or directory)\n```                                                                                                                                   18
**Syntax/Runtime Error**:\n```\nTraceback (most recent call last):\nValueError: invalid literal for int() with base 10: b''\n```                                                             